In [0]:
from pyspark.sql import functions as F
import re

base = "/Volumes/zomato_ai/bronze/raw_files"
datasets = ["orders", "restaurants", "reviews", "users", "food", "menu", "order_items"]

def clean(name):
    return re.sub(r"[^0-9a-zA-Z]+", "_", name.strip()).strip("_").lower()

failed = []
for name in datasets:
    try:
        raw = (spark.read
               .option("header", True)
               .option("inferSchema", False)
               .csv(f"{base}/{name}/"))

        df = raw.select(
            *[F.col(f"`{c}`").alias(clean(c)) for c in raw.columns],
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )

        (df.write.format("delta")
           .mode("overwrite")
           .option("overwriteSchema", True)
           .saveAsTable(f"zomato_ai.bronze.{name}"))

        rows = spark.table(f"zomato_ai.bronze.{name}").count()
        print(f"OK     {name}: {rows} rows")
    except Exception as e:
        failed.append(name)
        print(f"FAILED {name}: {str(e)[:300]}")

if failed:
    raise Exception(f"Bronze load failed for: {failed}")

OK     orders: 10000000 rows
OK     restaurants: 148544 rows
OK     reviews: 300000 rows
OK     users: 100000 rows
OK     food: 371632 rows
OK     menu: 1179936 rows
OK     order_items: 22998179 rows


In [0]:
for t in datasets:
    df = spark.table(f"zomato_ai.bronze.{t}")
    print(f"{t}: {df.count()} rows | columns: {df.columns}")

orders: 10000000 rows | columns: ['order_id', 'order_timestamp', 'order_date', 'user_id', 'r_id', 'restaurant_city', 'cuisine', 'items_count', 'sales_qty', 'subtotal', 'discount', 'delivery_fee', 'gst', 'sales_amount', 'currency', 'payment_method', 'order_status', 'customer_rating', 'delivery_time_min', '_source_file', '_ingested_at']
restaurants: 148544 rows | columns: ['c0', 'id', 'name', 'city', 'rating', 'rating_count', 'cost', 'cuisine', 'lic_no', 'link', 'address', 'menu', '_source_file', '_ingested_at']
reviews: 300000 rows | columns: ['review_id', 'order_id', 'user_id', 'restaurant_id', 'rating', 'comment', 'review_date', '_source_file', '_ingested_at']
users: 100000 rows | columns: ['c0', 'user_id', 'name', 'email', 'password', 'age', 'gender', 'marital_status', 'occupation', 'monthly_income', 'educational_qualifications', 'family_size', '_source_file', '_ingested_at']
food: 371632 rows | columns: ['c0', 'f_id', 'item', 'veg_or_non_veg', '_source_file', '_ingested_at']
menu: 1